# Built-in Tools

**Prerequisites:** all four earlier notebooks in this chapter

Every tool in this chapter so far has been a lie. `get_weather` read from a dict of three cities. `save_note` wrote to a dict. `list_customers` made up two hundred records with a loop. They were built to show one thing each, and they cooperated perfectly the rest of the time.

This notebook builds the toolbox an agent actually ends up with, against things that do not cooperate: a real HTTP API, a real disk, and a real expression evaluator. Each one carries a hazard worth meeting deliberately, once, rather than in production.

**A note on the outputs below.** One of these tools calls a live API over the network. What is recorded here is one run on one day. The numbers will differ when you run it, and the whole notebook needs an internet connection that the rest of the chapter did not.

**By the end you'll have:**
- Called a real API and watched notebook 3's machinery meet failures nobody wrote for it
- Seen a real response arrive at 7,254 tokens, in a shape nobody chose
- Confined file tools to one folder, and watched three ways out of it get refused
- Found out why checking a path's spelling is the version that gets bypassed
- Built a calculator that is not a way to run arbitrary Python, and seen that the obvious version is
- Met a tool that is dangerous without doing anything it wasn't asked to do
- Caught the observation step doing the agent's arithmetic for it, and closed it off twice: once by instruction, once with six lines that contain no model
- Named the one tool this chapter deliberately does not build

## Setup

In [1]:
import sys
import tempfile
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "02_agent_runtime"))

from chat import count_tokens
from invoke import invoke
from registry import Registry
from toolbox import calculate, file_tools, http_get, within, written_correctly
from verify import verified

## A tool that leaves the process

`http_get` is the first tool here whose failures are not ours to arrange. The endpoint is Open-Meteo, which needs no key and no account:

```python
@tool
def http_get(url: Annotated[str, "Full URL, including https://"]) -> str:
    """Fetch a web page or API response and return its body as text."""
    response = httpx.get(url, timeout=30, follow_redirects=True, headers=...)
    ...
    return response.text
```

In [2]:
FORECAST = ("https://api.open-meteo.com/v1/forecast"
            "?latitude=59.91&longitude=10.75&current=temperature_2m")

result = invoke(http_get, {"url": FORECAST})
print(result.kind, "|", count_tokens(result.content), "tokens")
print(result.content)

ok | 99 tokens
{"latitude":59.906296,"longitude":10.744095,"generationtime_ms":10.115742683410645,"utc_offset_seconds":0,"timezone":"GMT","timezone_abbreviation":"GMT","elevation":9.0,"current_units":{"time":"iso8601","interval":"seconds","temperature_2m":"°C"},"current":{"time":"2026-09-03T06:30","interval":900,"temperature_2m":13.2}}


That is the real version of the `get_weather` this chapter has been faking since notebook 1. Note what came back: not a temperature, but a JSON object with a nested `current` and a `generationtime_ms` nobody asked for.

**Nobody chose that shape.** Every fake tool in this chapter returned a tidy sentence, because the person writing the demo also wrote the tool. A real API returns what it returns, and the agent has to cope with it.

## When the endpoint doesn't cooperate

Notebook 3 split failures by who can act on them. Here is that split meeting failures it was not written for:

In [3]:
for url in ["https://api.open-meteo.com/v1/nonexistent",
            "https://not-a-real-host-xyzzy.example/data"]:
    result = invoke(http_get, {"url": url})
    print(f"{result.kind:9} | for the model: {result.for_the_model} | {result}")

refused   | for the model: True | https://api.open-meteo.com/v1/nonexistent returned HTTP 404 (Not Found).


refused   | for the model: True | Could not reach https://not-a-real-host-xyzzy.example/data: ConnectError. Check the address is right.


Both come back as `refused`, and only one of them is clear-cut.

**The 404 is easy.** The tool ran, the server answered, the answer was "no such thing". The model can act on that: try a different address. The status code is the part worth passing on, because 404, 429 and 500 imply three different next moves and nothing else distinguishes them.

**The dead host is a genuine grey area**, and it is worth admitting rather than tidying over. Notebook 3's rule says a failure is ours when the model cannot fix it — and a machine with no network is exactly that. But the commonest cause of an unreachable host, by a wide margin, is a URL the model invented. So it goes back as a refusal with the reason attached.

That is a judgment call, not a derivation. The rule from notebook 3 is a good rule and it has a grey zone; a real network is where you find it.

## When it cooperates too much

The same API, asked for sixteen days of hourly readings — which is one query parameter away from the one above:

In [4]:
BIG = FORECAST.replace("current=temperature_2m",
                       "hourly=temperature_2m,precipitation&forecast_days=16")

whole = invoke(http_get, {"url": BIG})
print("as it comes:", count_tokens(whole.content), "tokens")

cut = invoke(http_get, {"url": BIG}, max_tokens=150)
print("with max_tokens=150:", count_tokens(cut.content), "tokens")
print(str(cut).splitlines()[-1])

as it comes: 7181 tokens


with max_tokens=150: 167 tokens
[... cut short: 7,181 tokens in all, 150 kept]


Notebook 3 made this point with two hundred invented records. Here it is with a real endpoint, one parameter, and no bad intent anywhere: 7,254 tokens, which would sit in the conversation and be re-sent on every turn for the rest of the run.

The tool did nothing wrong. Neither did the model. `forecast_days=16` is a perfectly reasonable thing to ask for, and the size of the answer is not visible from the request.

## The disk

File tools come from a factory, not as module-level tools, because the folder they may touch is a decision about a particular agent rather than a property of the tool:

```python
read_file, write_file = file_tools(workspace)
```

Two agents in one process can hold different roots, and neither can reach the other's.

In [5]:
sandbox = Path(tempfile.mkdtemp())
(sandbox / "secrets.txt").write_text("PRETEND_API_KEY=hunter2", encoding="utf-8")

workspace = sandbox / "workspace"
read_file, write_file = file_tools(workspace)

print(invoke(write_file, {"path": "notes/deploy.md", "text": "Deploy is Friday."}))
print(invoke(read_file, {"path": "notes/deploy.md"}))

Wrote 17 characters to notes/deploy.md.
Deploy is Friday.


`secrets.txt` sits one level *above* the folder the agent may use, and holds an obviously fake key.

That is deliberate, and worth stating plainly: this repository has a real `.env` with a real Groq key in it, and pointing a path-traversal demo at your own credentials is how a key ends up committed inside a notebook's recorded output. The hazard is best demonstrated on something you made up.

In [6]:
for path in ["../secrets.txt", "notes/../../secrets.txt", "C:/Windows/win.ini"]:
    result = invoke(read_file, {"path": path})
    print(f"{path:26} -> {result.kind}: {result}")

../secrets.txt             -> refused: '../secrets.txt' is outside the folder this agent may use.
notes/../../secrets.txt    -> refused: 'notes/../../secrets.txt' is outside the folder this agent may use.
C:/Windows/win.ini         -> refused: 'C:/Windows/win.ini' is outside the folder this agent may use.


Three ways out, all refused, by one function:

```python
def within(root, path):
    root = Path(root).resolve()
    target = (root / path).resolve()
    if not target.is_relative_to(root):
        raise ToolError(f"'{path}' is outside the folder this agent may use.")
    return target
```

**`resolve()` is what does the work.** It flattens `..` and follows symlinks *first*, so the check is against where the path really lands rather than how it was written.

The version that gets bypassed is the one that checks the spelling — rejecting any path containing `..`. That looks equivalent and isn't, because there are always more spellings than you thought of: an absolute path contains no `..` at all, and neither does a symlink pointing out of the folder. Deciding what a path means is the filesystem's job, and it is better at it than a string check.

`write_file` also carries two things from earlier notebooks, and it is the tool that needed both:

- `repeatable=False`, from notebook 3. Writing twice is not writing once. The second write is harmless when the text is identical and destroys work when it isn't, and after a timeout there is no way to know which case you're in.
- a check from notebook 4, bound to the same root the tool was — because a check that reads a different folder than the one written to is worse than no check.

In [7]:
checked_write = verified(write_file, written_correctly(workspace))

print(invoke(checked_write, {"path": "notes/release.md", "text": "Ship on Friday."}).kind)
print("repeatable:", write_file.repeatable)

ok
repeatable: False


## Arithmetic

Models are bad at arithmetic, so a calculator is one of the first tools anyone reaches for. It is also the shortest path from "my agent has tools" to "my agent runs arbitrary code", because the obvious implementation is one line:

```python
def calculate(expression):
    return str(eval(expression))
```

In [8]:
print(eval("(3 + 4) * 2"))
print(eval("__import__('os').getcwd()"))

14
c:\Users\91879\OneDrive\AI Projects (Venkat)\agentic-forge-cookbook\03_tools\notebooks


The second line is not arithmetic. It is a shell.

`eval` takes any Python expression, and an expression can import modules, read files, open sockets and start processes. A model that has been talked into sending that string — by a web page it fetched, by a document it summarized, by a user — has full run of the process. Chapter 1's notebook 4 demonstrated prompt injection; this is what injection is *for*.

The fix is not a list of banned words. `calculate` parses the expression and walks the tree, handling the node types it knows and refusing everything else:

```python
def _value_of(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _BINARY:
        ...
    raise ToolError(f"... is not arithmetic. Only numbers and + - * / // % ** are allowed.")
```

An allowlist has to be right about what it permits. A blocklist has to be right about everything anyone will ever try, and only needs to be wrong once.

In [9]:
for expression in ["(3 + 4) * 2", "10 / 4", "2 ** 10",
                   "__import__('os').getcwd()", "print('hi')", "3 +", "9 ** 9 ** 9"]:
    result = invoke(calculate, {"expression": expression})
    print(f"{expression:26} -> {result.kind:8} {str(result)[:52]}")

(3 + 4) * 2                -> ok       14
10 / 4                     -> ok       2.5
2 ** 10                    -> ok       1024
__import__('os').getcwd()  -> refused  'Call(func=Attribute(value=Call(func=Name...' is not
print('hi')                -> refused  'Call(func=Name(id='print', ctx=Load()), ...' is not
3 +                        -> refused  '3 +' is not a complete expression: invalid syntax.
9 ** 9 ** 9                -> refused  An exponent of 387420489 is too large to work out. K


The last one is the interesting one.

`9 ** 9 ** 9` is arithmetic. Every node in it is on the allowlist, nothing is imported, nothing is read, and the tool is being asked to do precisely the job it exists to do. It would also occupy a core for longer than anyone will wait, and notebook 3's timeout cannot stop it — that timeout frees the *agent*, while the thread keeps computing.

**A tool can be dangerous without doing anything it was not asked to do.** So `calculate` refuses an exponent over 100, which is a limit on the work rather than on the syntax.

## The tool this chapter does not build

Every real agent toolbox eventually grows a `run_python` or a `run_shell`, and this chapter does not build one.

Not out of squeamishness — because everything above stops working at that point. Confining `read_file` to a folder is worth doing precisely because `open()` is the only way that tool can touch the disk. A tool that runs arbitrary Python has `open()`, and `socket`, and `subprocess`, and can rewrite the very check that was supposed to contain it. `within()` protects nothing once the code being run is chosen by the model.

The same goes for the rest of it. An allowlist of AST nodes is meaningless when the whole language is the point. `repeatable=False` describes a call whose effects you can at least name. A timeout does not stop a thread.

Running model-written code safely needs a boundary the process itself cannot cross — a separate process, a container, a virtual machine — and that is `09_sandbox`. It is a different kind of problem from anything in this chapter, and pretending otherwise by adding a `run_python` with a blocklist would be the most dangerous thing in the repo.

## Into the loop

The whole toolbox, in chapter 2's ReAct loop, against a live endpoint. The question needs the network and the calculator:

In [10]:
from chat import Conversation
from runtime import Runtime
from observe import Observe
from think import Think
from act import Act

toolbox = Registry([http_get, calculate, read_file, checked_write])
print(toolbox)

runtime = Runtime(
    loop=[
        Observe(event=f"Fetch {FORECAST} and tell me the current temperature in Oslo "
                      "in Fahrenheit. Use the calculator for the conversion."),
        Think(tools=toolbox.tools, allow_fork=False),
        Act(functions=toolbox.invokers(timeout=30, max_tokens=400), schemas=None),
        Observe(),
    ],
    repeat_from=1,
    max_iterations=10,
)

conversation = runtime.run(Conversation())
for m in conversation.messages:
    print(f"- {m['role']:9} {str(m['content'])[:150]!r}")

<Registry 4 tools: http_get, calculate, read_file, write_file>


- user      'Fetch https://api.open-meteo.com/v1/forecast?latitude=59.91&longitude=10.75&current=temperature_2m and tell me the current temperature in Oslo in Fahr'
- decision  '{"url":"https://api.open-meteo.com/v1/forecast?latitude=59.91\\u0026longitude=10.75\\u0026current=temperature_2m"}'
- tool      'The current temperature in Oslo is 13.2°C.'
- decision  '{"expression":"(13.2 * 9/5) + 32"}'
- tool      'The current temperature in Oslo is 55.76\u202f°F.'
- decision  'The current temperature in Oslo is 55.76\u202f°F.'
- assistant 'The current temperature in Oslo is 55.76\u202f°F.'


`calculate` is called, with `(13.2 * 9/5) + 32`, and the observation before it reports 13.2 °C — the number and the unit the API actually returned.

It did not do that the first time this notebook was run. That transcript went:

```
- tool      'The current temperature in Oslo is 68 °F.'
- decision  '68 °F.'
```

with no call to `calculate` anywhere. The API had returned `"temperature_2m": 20.0` alongside `"temperature_2m": "°C"`. And 20 °C is exactly 68 °F, so the conversion had happened — inside `Observe`.

`Observe` has no tools. It makes one LLM call to turn a raw tool result into a sentence, and that call did arithmetic the agent had a tool for.

**The answer was correct, which is what made it worth stopping on.** A wrong one gets noticed. That one was right, reached by a path nobody designed, with the sum recorded nowhere — and notebook 4's checks cannot help, because they check tool calls and this was never one.

The cause was one clause in the prompt. It asked for the result "using the conversation only to phrase it correctly (e.g. which units, which entity)". That was written to stop a value being mislabelled. Read with a request for Fahrenheit sitting in the conversation, it also reads as permission to restate the value in those units.

`02_agent_runtime/observe.py` now says the opposite, as plainly as it can:

> Copy every value exactly as the tool gave it, in the units the tool gave it. Do not convert, calculate, round, combine or reformat anything — not even when the conversation asks for a different unit or form.

The transcript above is that prompt running. The 13.2 survives as 13.2, the model sees it was asked for Fahrenheit, and it reaches for the tool.

## Taking the phrasing step out

An extractor is a plain Python function that turns a raw tool result into an observation, with no model involved. Chapter 2 offered them as a cost saving. The stricter prompt above already fixed this particular run, so what follows is not the fix — it is the same thing bought with a stronger guarantee:

In [11]:
import json

def temperature_only(effect):
    try:
        data = json.loads(effect["content"])
        current, units = data["current"], data["current_units"]
        return (f"http_get returned temperature_2m = {current['temperature_2m']} "
                f"{units['temperature_2m']} at {current['time']}.")
    except (ValueError, KeyError):
        return effect["content"]      # not the JSON we expected -- hand it back untouched

exact = Runtime(
    loop=[
        Observe(event=f"Fetch {FORECAST} and tell me the current temperature in Oslo "
                      "in Fahrenheit. Use the calculator for the conversion."),
        Think(tools=toolbox.tools, allow_fork=False),
        Act(functions=toolbox.invokers(timeout=30, max_tokens=400), schemas=None),
        Observe(extractors={"http_get": temperature_only}),
    ],
    repeat_from=1,
    max_iterations=10,
)

for m in exact.run(Conversation()).messages:
    print(f"- {m['role']:9} {str(m['content'])[:150]!r}")

- user      'Fetch https://api.open-meteo.com/v1/forecast?latitude=59.91&longitude=10.75&current=temperature_2m and tell me the current temperature in Oslo in Fahr'
- decision  '{"url":"https://api.open-meteo.com/v1/forecast?latitude=59.91\\u0026longitude=10.75\\u0026current=temperature_2m"}'
- tool      'http_get returned temperature_2m = 13.2 °C at 2026-09-03T06:30.'
- decision  '{"expression":"(13.2 * 9/5) + 32"}'
- tool      'The calculation result is 55.76.'
- decision  'The current temperature in Oslo is 55.76\u202f°F.'
- assistant 'The current temperature in Oslo is 55.76\u202f°F.'


The same answer, reached without asking a model to behave.

The extractor is six lines of Python. It cannot decide to be helpful, cannot be talked round by something sitting in the conversation, and costs nothing to run. The stricter prompt gets there by instruction — and an instruction is followed *most* of the time.

That is the real difference between the two fixes, and it is worth knowing which one you have. The prompt protects every tool whose output shape you cannot predict, which is most of them. The extractor is what you reach for once a tool's response has a known shape and the value matters. Here it also reads better, because it names the field and the timestamp instead of describing them.

This chapter has made that trade several times now. A rule the model is *asked* to follow — the tool descriptions in notebook 1, the failure messages in notebook 3, this prompt — against a rule it *cannot break*: the generated argument check, `within()`, the parsed calculator. Both are worth having. Only the second kind is a guarantee.

## What you built

✓ Called a real API and got back a nested JSON object with a `generationtime_ms` in it — the first tool in this chapter whose output shape nobody chose

✓ Watched notebook 3's split by who-can-act meet a real 404 and a real dead host, and found the grey zone: an unreachable host is ours by the rule and the model's in practice, so it goes back as a refusal with the reason attached

✓ Measured a live response at 7,254 tokens, one query parameter away from a 100-token one, with nothing wrong on either side

✓ Confined file tools to a folder with `within()`, and refused a relative escape, a nested escape and an absolute path with the same three lines

✓ Learned why `resolve()` beats checking the spelling: an absolute path contains no `..`, and neither does a symlink

✓ Ran the traversal demo against a made-up secret in a temp folder, on purpose, because pointing it at this repo's real `.env` is how a key gets committed inside a notebook's output

✓ Gave `write_file` the two flags it needed and nothing else did: `repeatable=False` from notebook 3, and a notebook 4 check bound to the same root

✓ Saw `eval("__import__('os').getcwd()")` work, and replaced the one-line calculator with a walk over parsed nodes — an allowlist has to be right about what it permits, a blocklist about everything anyone will ever try

✓ Met `9 ** 9 ** 9`: entirely arithmetic, entirely allowed, and a denial of service — so `calculate` limits the work and not just the syntax

✓ Left `run_python` unbuilt, and said why: every protection in this chapter assumes the tool can only do the one thing it names

✓ Caught `Observe` converting 20 °C to 68 °F inside its phrasing call, so `calculate` was never used — a correct answer reached by a path nobody designed, with the arithmetic recorded nowhere and nothing in notebook 4 able to check it

✓ Traced it to one clause meant to prevent mislabelling, and tightened the prompt in `02_agent_runtime/observe.py` to forbid changing a value at all — after which the model called `calculate` itself

✓ Did it a second way with a deterministic extractor, and named the difference: a prompt is a rule the model is asked to follow, while a function with no model in it cannot decide to be helpful

## The chapter, end to end

A tool started as three copies of one fact that could drift apart. It became one object with the schema and the argument check read off the signature; then a registry, once there were too many to keep in three hand-written dicts; then a call boundary with a time limit, a size limit and an ending for every way it can go wrong; then a check that looks at the world rather than at what the tool claimed. This notebook pointed all of it at things that do not cooperate.

What the chapter does not do is decide *which* tools an agent should be given for a particular request. Every subset here was chosen by hand, by someone who already knew the question.

**Next:** `04_memory` — what an agent keeps between turns, and between runs.